# Using Laurium's `Extractor` with Ollama

In this notebook, we'll examine how to use Laurium's `Extractor` class with a
locally-running Ollama model. To begin, install [Ollama](https://ollama.com/),
make sure it's running, and pull the model used in this notebook:

```
ollama pull qwen2.5:7b
```

In [1]:
from typing import Literal

import pandas as pd

from laurium.decoder_models import llm
from laurium.decoder_models.extractor import Extractor

## Dataset

We'll be looking at free-text data from a feedback and support form, with the goal of triaging and categorising the data.

In [2]:
# ruff: ignore[E501]
feedback_data = pd.DataFrame(
    {
        "comment": [
            "The login system crashed and I lost all my work!",
            "Really appreciate the new dark mode feature",
            "Can we get a mobile app version soon?",
            "Billing charged me twice this month, need help",
            "The password reset link keeps expiring before I can use it",
            "Your sales team was incredibly helpful during our renewal",
            "I haven't noticed much difference since the latest update",
            "Please remove me from these marketing emails immediately",
            "The new campaign looks great and the messaging is really clear",
            "I'm concerned that my account showed a login from an unknown device",
            "The dashboard takes a little longer to load, but it still works",
            "Could someone from sales send me pricing for the enterprise plan?",
            "The security verification process was straightforward and reassuring",
            "I submitted a support request to IT three days ago and still have no response",
            "The webinar was useful, although some of the examples felt repetitive",
            "Our integration has stopped syncing data and this is blocking our team",
            "Thanks for resolving my invoice issue so quickly!",
            "I'd like to update the contact details associated with my account",
            "The promotional offer was confusing and I couldn't tell when it expired",
            "I think someone may have accessed my account—please investigate urgently",
        ]
    }
)
feedback_data.head(5)

,comment
0,The login system crashed and I lost all my work!
1,Really appreciate the new dark mode feature
2,Can we get a mobile app version soon?
3,"Billing charged me twice this month, need help"
4,The password reset link keeps expiring before ...


## Configuring our extraction pipeline

We'll start by configuring our LLM to use Ollama, running locally.

In [3]:
# Create LLM instance
feedback_llm = llm.create_llm(
    llm_platform="ollama",
    model_name="qwen2.5:7b",
    temperature=0.0,  # To reduce stochasticity in responses
)

### Setting up a classification model

We want to answer the following four questions for each comment:
- Is the input positive, negative or neutral in tone?
- How urgent is this request on a 3-point scale?
- Which department should this comment be routed to?
- Is follow-up action required?

To do this, we'll set up a classification schema with descriptions of each field.

In [4]:
schema = {
    "sentiment": (
        Literal["positive", "negative", "neutral"],
        "Customer's emotional tone",
    ),
    "urgency": (
        Literal[1, 2, 3],  # 1-3 scale
        "How quickly this needs attention (1=low, 3=urgent)",
    ),
    "department": (
        Literal["IT", "Security", "Product", "Sales", "Other"],
        "Which department should handle this",
    ),
    "action_required": (
        Literal["yes", "no"],
        "Whether immediate action is needed",
    ),
}

We can also give the model some examples of how we would classify inputs, which
can improve the performance of the model (this is known as [few-shot
prompting](https://www.promptingguide.ai/techniques/fewshot)).

In [5]:
few_shot_examples = [
    {
        "text": "System is down, can't access anything!",
        "sentiment": "negative",
        "urgency": 3,
        "department": "IT",
        "action_required": "yes",
    },
    {
        "text": "Love the new interface design",
        "sentiment": "positive",
        "urgency": 1,
        "department": "Product",
        "action_required": "no",
    },
]

### Setting up the extractor

Now we'll tell the LLM what we want it to do, and build our `Extractor`,
which takes care of creating the output schema, prompt and parser for us.

In [6]:
# Create the extractor - this builds the prompt, output parser and LLM chain
extractor = Extractor(
    schema,
    feedback_llm,
    prompt="Analyze customer feedback and extract structured information.",
    keywords=["urgent", "complaint", "praise", "bug", "feature"],
    examples=few_shot_examples,
    example_human_template="Feedback: {text}",
    example_assistant_template="""{{
        "sentiment": "{sentiment}",
        "urgency": {urgency},
        "department": "{department}",
        "action_required": "{action_required}"
    }}""",
)

We can see the full instructions that we're providing to the LLM

In [7]:
# Examine the prompt to see what it looks like
print(extractor.prompt.messages[0].prompt.template)

Analyze customer feedback and extract structured information.
Pay special attention to these keywords: urgent, complaint, praise, bug, feature

For each text, extract:
    - sentiment: Customer's emotional tone
    - urgency: How quickly this needs attention (1=low, 3=urgent)
    - department: Which department should handle this
    - action_required: Whether immediate action is needed

Expected output format:
{{
    "sentiment": "positive"|"negative"|"neutral",
    "urgency": 1|2|3,
    "department": "IT"|"Security"|"Product"|"Sales"|"Other",
    "action_required": "yes"|"no"
}}


## Results

We're now ready to apply the batch extractor to our free text!

In [ ]:
# Should take 20-40 seconds to run
results = extractor.batch_label(feedback_data, text_column="comment")
results

,comment,sentiment,urgency,department,action_required
0,The login system crashed and I lost all my work!,negative,3,IT,yes
1,Really appreciate the new dark mode feature,positive,1,Product,no
2,Can we get a mobile app version soon?,neutral,2,Product,yes
3,"Billing charged me twice this month, need help",negative,2,Sales,yes
4,The password reset link keeps expiring before ...,negative,2,IT,yes
5,Your sales team was incredibly helpful during ...,positive,1,Sales,no
6,I haven't noticed much difference since the la...,neutral,1,Product,no
7,Please remove me from these marketing emails i...,negative,2,Sales,yes
8,The new campaign looks great and the messaging...,positive,1,Sales,no
9,I'm concerned that my account showed a login f...,negative,2,Security,yes
